In [1]:
import pdfplumber
import pytesseract
from PIL import Image, ImageDraw, ImageFont
import pyttsx3
import os
import moviepy.editor as mpy
from moviepy.editor import TextClip, AudioFileClip

# Set the path to your Tesseract executable
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"  # Update this path

# Function to extract text from the PDF using pdfplumber or OCR if text is not extractable
def extract_text_from_pdf(pdf_path):
    try:
        text = ''
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages):
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
                else:
                    print(f"Warning: No text extracted from page {page_num + 1}, using OCR...")
                    page_image = page.to_image()
                    page_text = pytesseract.image_to_string(page_image.original)
                    text += page_text + "\n"
        return text
    except FileNotFoundError:
        print(f"Error: The PDF file at {pdf_path} was not found.")
        return ""
    except Exception as e:
        print(f"Error extracting text from PDF: {e}")
        return ""

# Function to convert text to speech and save as an audio file
def text_to_speech(text, output_audio_path):
    try:
        engine = pyttsx3.init()
        engine.setProperty('rate', 150)
        engine.setProperty('volume', 1)
        
        # Save speech to an audio file
        engine.save_to_file(text, output_audio_path)
        engine.runAndWait()
        print("Text-to-speech conversion completed and audio file saved.")
    except Exception as e:
        print(f"Error during text-to-speech conversion: {e}")

# Function to generate video slides from text
def create_video_from_text(text, output_video_path, audio_file_path):
    try:
        # Split text into segments (for simplicity, each line is a segment)
        text_lines = text.split('\n')
        
        # Create individual slides for each text line
        clips = []
        for idx, line in enumerate(text_lines):
            if line.strip() == "":
                continue  # Skip empty lines
            
            # Create an image for each text line
            image = Image.new('RGB', (1280, 720), color=(255, 255, 255))  # White background
            draw = ImageDraw.Draw(image)
            font = ImageFont.load_default()  # You can specify a different font here
            draw.text((50, 200), line, fill="black", font=font)  # Positioning text on image
            
            # Convert image to video clip
            clip = mpy.ImageClip(image).set_duration(3)  # Each slide lasts 3 seconds
            clips.append(clip)
        
        # Concatenate all slides into one video
        video = mpy.concatenate_videoclips(clips, method="compose")
        
        # Load audio file and set it to match the video duration
        audio = AudioFileClip(audio_file_path)
        video = video.set_audio(audio)
        
        # Write the final video to a file
        video.write_videofile(output_video_path, fps=24)
        print("Video creation completed successfully.")
    except Exception as e:
        print(f"Error during video creation: {e}")

# Main function to execute the process
def main(pdf_path, output_audio_path, output_video_path):
    text = extract_text_from_pdf(pdf_path)
    if text:
        print("Text extracted successfully, now converting to speech...")
        text_to_speech(text, output_audio_path)
        
        print("Creating video from text...")
        create_video_from_text(text, output_video_path, output_audio_path)
    else:
        print("No text found in the PDF.")

# File paths
pdf_path = r"C:\Users\ASUS\Downloads\Graph Theory (Part 03) (Lec 09) _ Class Notes.pdf"
audio_file_path = r"C:\Users\ASUS\Downloads\output_audio.mp3"
video_file_path = r"C:\Users\ASUS\Downloads\output_video.mp4"

# Check if the PDF file exists before proceeding
if not os.path.exists(pdf_path):
    print(f"Error: The PDF file at {pdf_path} does not exist.")
else:
    # Execute the main function
    main(pdf_path, audio_file_path, video_file_path)


ModuleNotFoundError: No module named 'moviepy.editor'

In [2]:
pip install moviepy


Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install pdfplumber pytesseract pyttsx3 pillow opencv-python pydub numpy


Note: you may need to restart the kernel to use updated packages.


In [6]:
import pdfplumber
import pytesseract
from PIL import Image, ImageDraw, ImageFont
import pyttsx3
import os
import cv2
import numpy as np
from pydub import AudioSegment

# Set the path to your Tesseract executable
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"  # Update this path

# Function to extract text from the PDF using pdfplumber or OCR if text is not extractable
def extract_text_from_pdf(pdf_path):
    try:
        text = ''
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages):
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
                else:
                    print(f"Warning: No text extracted from page {page_num + 1}, using OCR...")
                    page_image = page.to_image()
                    page_text = pytesseract.image_to_string(page_image.original)
                    text += page_text + "\n"
        return text
    except FileNotFoundError:
        print(f"Error: The PDF file at {pdf_path} was not found.")
        return ""
    except Exception as e:
        print(f"Error extracting text from PDF: {e}")
        return ""

# Function to convert text to speech and save as an audio file
def text_to_speech(text, output_audio_path):
    try:
        engine = pyttsx3.init()
        engine.setProperty('rate', 150)
        engine.setProperty('volume', 1)
        
        # Save speech to an audio file
        engine.save_to_file(text, output_audio_path)
        engine.runAndWait()
        print("Text-to-speech conversion completed and audio file saved.")
    except Exception as e:
        print(f"Error during text-to-speech conversion: {e}")

# Function to create video from images using OpenCV
def create_video_from_images(text, video_output_path, audio_output_path, image_duration=3):
    try:
        # Split text into segments (for simplicity, each line is a segment)
        text_lines = text.split('\n')
        
        # Create the images for each line of text
        images = []
        for idx, line in enumerate(text_lines):
            if line.strip() == "":
                continue  # Skip empty lines
            
            # Create an image for each text line
            image = Image.new('RGB', (1280, 720), color=(255, 255, 255))  # White background
            draw = ImageDraw.Draw(image)
            font = ImageFont.load_default()  # You can specify a different font here
            draw.text((50, 200), line, fill="black", font=font)  # Positioning text on image
            
            # Convert the PIL image to an OpenCV format (BGR)
            opencv_image = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
            images.append(opencv_image)
        
        # Create a video from the images
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Define video codec (MP4)
        video_out = cv2.VideoWriter(video_output_path, fourcc, 1.0, (1280, 720))  # 1 FPS to match 1-second per slide
        
        for img in images:
            video_out.write(img)  # Write each image to the video
            
        # Close video writer
        video_out.release()
        print(f"Video created successfully: {video_output_path}")
        
        # Add audio to the video
        add_audio_to_video(video_output_path, audio_output_path)
    
    except Exception as e:
        print(f"Error during video creation: {e}")

# Function to add audio to the video (using pydub and ffmpeg)
def add_audio_to_video(video_path, audio_path):
    try:
        # Convert the audio file to a format supported by ffmpeg (e.g., WAV if necessary)
        audio = AudioSegment.from_mp3(audio_path)
        audio.export("temp_audio.wav", format="wav")

        # Using ffmpeg to add audio to video
        os.system(f"ffmpeg -i {video_path} -i temp_audio.wav -c:v copy -c:a aac -strict experimental {video_path.replace('.mp4', '_with_audio.mp4')}")

        print("Audio added to the video successfully.")
        
        # Clean up temporary audio file
        os.remove("temp_audio.wav")
    except Exception as e:
        print(f"Error during audio addition: {e}")

# Main function to execute the process
def main(pdf_path, output_audio_path, output_video_path):
    text = extract_text_from_pdf(pdf_path)
    if text:
        print("Text extracted successfully, now converting to speech...")
        text_to_speech(text, output_audio_path)
        
        print("Creating video from text...")
        create_video_from_images(text, output_video_path, output_audio_path)
    else:
        print("No text found in the PDF.")

# File paths
pdf_path = r"C:\Users\ASUS\Downloads\Graph Theory (Part 03) (Lec 09) _ Class Notes.pdf"
audio_file_path = r"C:\Users\ASUS\Downloads\output_audio.mp3"
video_file_path = r"C:\Users\ASUS\Downloads\output_video.mp4"

# Check if the PDF file exists before proceeding
if not os.path.exists(pdf_path):
    print(f"Error: The PDF file at {pdf_path} does not exist.")
else:
    # Execute the main function
    main(pdf_path, audio_file_path, video_file_path)


Text extracted successfully, now converting to speech...
Text-to-speech conversion completed and audio file saved.
Creating video from text...
Video created successfully: C:\Users\ASUS\Downloads\output_video.mp4
Error during audio addition: [WinError 2] The system cannot find the file specified


C:\Users\ASUS\anaconda3\Lib\site-packages\pydub\utils.py:198: RuntimeWarning: Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work
  warn("Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work", RuntimeWarning)


In [8]:
import pdfplumber
import pytesseract
from PIL import Image, ImageDraw, ImageFont
import pyttsx3
import os
import cv2
import numpy as np
from pydub import AudioSegment

# Set the path to your Tesseract executable
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"  # Update this path

# Path to ffmpeg (either add it to your PATH or specify the full path here)
FFMPEG_PATH = r"C:\ffmpeg\bin\ffmpeg.exe"  # Adjust this path to where ffmpeg is installed

# Function to extract text from the PDF using pdfplumber or OCR if text is not extractable
def extract_text_from_pdf(pdf_path):
    try:
        text = ''
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages):
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
                else:
                    print(f"Warning: No text extracted from page {page_num + 1}, using OCR...")
                    page_image = page.to_image()
                    page_text = pytesseract.image_to_string(page_image.original)
                    text += page_text + "\n"
        return text
    except FileNotFoundError:
        print(f"Error: The PDF file at {pdf_path} was not found.")
        return ""
    except Exception as e:
        print(f"Error extracting text from PDF: {e}")
        return ""

# Function to convert text to speech and save as an audio file
def text_to_speech(text, output_audio_path):
    try:
        engine = pyttsx3.init()
        engine.setProperty('rate', 150)
        engine.setProperty('volume', 1)
        
        # Save speech to an audio file
        engine.save_to_file(text, output_audio_path)
        engine.runAndWait()
        print("Text-to-speech conversion completed and audio file saved.")
    except Exception as e:
        print(f"Error during text-to-speech conversion: {e}")

# Function to create video from images using OpenCV
def create_video_from_images(text, video_output_path, audio_output_path, image_duration=3):
    try:
        # Split text into segments (for simplicity, each line is a segment)
        text_lines = text.split('\n')
        
        # Create the images for each line of text
        images = []
        for idx, line in enumerate(text_lines):
            if line.strip() == "":
                continue  # Skip empty lines
            
            # Create an image for each text line
            image = Image.new('RGB', (1280, 720), color=(255, 255, 255))  # White background
            draw = ImageDraw.Draw(image)
            font = ImageFont.load_default()  # You can specify a different font here
            draw.text((50, 200), line, fill="black", font=font)  # Positioning text on image
            
            # Convert the PIL image to an OpenCV format (BGR)
            opencv_image = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
            images.append(opencv_image)
        
        # Create a video from the images
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Define video codec (MP4)
        video_out = cv2.VideoWriter(video_output_path, fourcc, 1.0, (1280, 720))  # 1 FPS to match 1-second per slide
        
        for img in images:
            video_out.write(img)  # Write each image to the video
            
        # Close video writer
        video_out.release()
        print(f"Video created successfully: {video_output_path}")
        
        # Add audio to the video
        add_audio_to_video(video_output_path, audio_output_path)
    
    except Exception as e:
        print(f"Error during video creation: {e}")

# Function to add audio to the video (using pydub and ffmpeg)
def add_audio_to_video(video_path, audio_path):
    try:
        # Check if FFMPEG_PATH is correctly set, and update if necessary
        if not os.path.exists(FFMPEG_PATH):
            print(f"Error: ffmpeg not found at {FFMPEG_PATH}")
            return

        # Convert the audio file to a format supported by ffmpeg (e.g., WAV if necessary)
        audio = AudioSegment.from_mp3(audio_path)
        audio.export("temp_audio.wav", format="wav")

        # Using ffmpeg to add audio to video
        os.system(f'"{FFMPEG_PATH}" -i {video_path} -i temp_audio.wav -c:v copy -c:a aac -strict experimental {video_path.replace(".mp4", "_with_audio.mp4")}')
        
        print("Audio added to the video successfully.")
        
        # Clean up temporary audio file
        os.remove("temp_audio.wav")
    except Exception as e:
        print(f"Error during audio addition: {e}")

# Main function to execute the process
def main(pdf_path, output_audio_path, output_video_path):
    text = extract_text_from_pdf(pdf_path)
    if text:
        print("Text extracted successfully, now converting to speech...")
        text_to_speech(text, output_audio_path)
        
        print("Creating video from text...")
        create_video_from_images(text, output_video_path, output_audio_path)
    else:
        print("No text found in the PDF.")

# File paths
pdf_path = r"C:\Users\ASUS\Downloads\Graph Theory (Part 03) (Lec 09) _ Class Notes.pdf"
audio_file_path = r"C:\Users\ASUS\Downloads\output_audio.mp3"
video_file_path = r"C:\Users\ASUS\Downloads\output_video.mp4"

# Check if the PDF file exists before proceeding
if not os.path.exists(pdf_path):
    print(f"Error: The PDF file at {pdf_path} does not exist.")
else:
    # Execute the main function
    main(pdf_path, audio_file_path, video_file_path)


Text extracted successfully, now converting to speech...
Text-to-speech conversion completed and audio file saved.
Creating video from text...
Video created successfully: C:\Users\ASUS\Downloads\output_video.mp4
Error: ffmpeg not found at C:\ffmpeg\bin\ffmpeg.exe


In [9]:
import pdfplumber
import pytesseract
from PIL import Image, ImageDraw, ImageFont
import pyttsx3
import os
import cv2
import numpy as np
from pydub import AudioSegment
import time

# Set the path to your Tesseract executable
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"  # Update this path

# Function to extract text from the PDF using pdfplumber or OCR if text is not extractable
def extract_text_from_pdf(pdf_path):
    try:
        text = ''
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages):
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
                else:
                    print(f"Warning: No text extracted from page {page_num + 1}, using OCR...")
                    page_image = page.to_image()
                    page_text = pytesseract.image_to_string(page_image.original)
                    text += page_text + "\n"
        return text
    except FileNotFoundError:
        print(f"Error: The PDF file at {pdf_path} was not found.")
        return ""
    except Exception as e:
        print(f"Error extracting text from PDF: {e}")
        return ""

# Function to convert text to speech and save as an audio file
def text_to_speech(text, output_audio_path):
    try:
        engine = pyttsx3.init()
        engine.setProperty('rate', 150)
        engine.setProperty('volume', 1)
        
        # Save speech to an audio file
        engine.save_to_file(text, output_audio_path)
        engine.runAndWait()
        print("Text-to-speech conversion completed and audio file saved.")
    except Exception as e:
        print(f"Error during text-to-speech conversion: {e}")

# Function to create video from images using OpenCV and add captions
def create_video_from_images_with_captions(text, video_output_path, audio_output_path, image_duration=3):
    try:
        # Split text into segments (for simplicity, each line is a segment)
        text_lines = text.split('\n')
        
        # Create the images for each line of text (captions)
        images = []
        timings = []
        for idx, line in enumerate(text_lines):
            if line.strip() == "":
                continue  # Skip empty lines
            
            # Create an image for each text line
            image = Image.new('RGB', (1280, 720), color=(255, 255, 255))  # White background
            draw = ImageDraw.Draw(image)
            font = ImageFont.load_default()  # You can specify a different font here
            draw.text((50, 200), line, fill="black", font=font)  # Positioning text on image
            
            # Convert the PIL image to an OpenCV format (BGR)
            opencv_image = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
            images.append(opencv_image)
            
            # Timings for each caption, assuming each caption stays for 1 second
            timings.append(time.time() + idx * image_duration)
        
        # Create a video from the images
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Define video codec (MP4)
        video_out = cv2.VideoWriter(video_output_path, fourcc, 1.0, (1280, 720))  # 1 FPS to match 1-second per slide
        
        for img in images:
            video_out.write(img)  # Write each image to the video
            
        # Close video writer
        video_out.release()
        print(f"Video created successfully: {video_output_path}")
        
        # Add audio to the video
        add_audio_to_video(video_output_path, audio_output_path)
    
    except Exception as e:
        print(f"Error during video creation: {e}")

# Function to add audio to the video (using pydub and ffmpeg)
def add_audio_to_video(video_path, audio_path):
    try:
        # Convert the audio file to a format supported by ffmpeg (e.g., WAV if necessary)
        audio = AudioSegment.from_mp3(audio_path)
        audio.export("temp_audio.wav", format="wav")

        # Using ffmpeg to add audio to video
        os.system(f"ffmpeg -i {video_path} -i temp_audio.wav -c:v copy -c:a aac -strict experimental {video_path.replace('.mp4', '_with_audio.mp4')}")

        print("Audio added to the video successfully.")
        
        # Clean up temporary audio file
        os.remove("temp_audio.wav")
    except Exception as e:
        print(f"Error during audio addition: {e}")

# Main function to execute the process
def main(pdf_path, output_audio_path, output_video_path):
    text = extract_text_from_pdf(pdf_path)
    if text:
        print("Text extracted successfully, now converting to speech...")
        text_to_speech(text, output_audio_path)
        
        print("Creating video from text with captions...")
        create_video_from_images_with_captions(text, output_video_path, output_audio_path)
    else:
        print("No text found in the PDF.")

# File paths
pdf_path = r"C:\Users\ASUS\Downloads\Graph Theory (Part 03) (Lec 09) _ Class Notes.pdf"
audio_file_path = r"C:\Users\ASUS\Downloads\output_audio.mp3"
video_file_path = r"C:\Users\ASUS\Downloads\output_video.mp4"

# Check if the PDF file exists before proceeding
if not os.path.exists(pdf_path):
    print(f"Error: The PDF file at {pdf_path} does not exist.")
else:
    # Execute the main function
    main(pdf_path, audio_file_path, video_file_path)


Text extracted successfully, now converting to speech...
Text-to-speech conversion completed and audio file saved.
Creating video from text with captions...
Video created successfully: C:\Users\ASUS\Downloads\output_video.mp4
Error during audio addition: [WinError 2] The system cannot find the file specified


In [10]:
import pdfplumber
import pytesseract
from PIL import Image, ImageDraw, ImageFont
import pyttsx3
import os
import cv2
import numpy as np
from pydub import AudioSegment
import time

# Set the path to your Tesseract executable
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"  # Update this path

# Function to extract text from the PDF using pdfplumber or OCR if text is not extractable
def extract_text_from_pdf(pdf_path):
    try:
        text = ''
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages):
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
                else:
                    print(f"Warning: No text extracted from page {page_num + 1}, using OCR...")
                    page_image = page.to_image()
                    page_text = pytesseract.image_to_string(page_image.original)
                    text += page_text + "\n"
        return text
    except FileNotFoundError:
        print(f"Error: The PDF file at {pdf_path} was not found.")
        return ""
    except Exception as e:
        print(f"Error extracting text from PDF: {e}")
        return ""

# Function to convert text to speech and save as an audio file
def text_to_speech(text, output_audio_path):
    try:
        engine = pyttsx3.init()
        engine.setProperty('rate', 150)
        engine.setProperty('volume', 1)
        
        # Save speech to an audio file
        engine.save_to_file(text, output_audio_path)
        engine.runAndWait()
        print("Text-to-speech conversion completed and audio file saved.")
    except Exception as e:
        print(f"Error during text-to-speech conversion: {e}")

# Function to create video from images using OpenCV and add captions
def create_video_from_images_with_captions(text, video_output_path, audio_output_path, image_duration=3):
    try:
        # Split text into segments (for simplicity, each line is a segment)
        text_lines = text.split('\n')
        
        # Create the images for each line of text (captions)
        images = []
        timings = []
        for idx, line in enumerate(text_lines):
            if line.strip() == "":
                continue  # Skip empty lines
            
            # Create an image for each line of text (for simplicity, we'll use a blank image here, you can replace it with actual content)
            image = Image.new('RGB', (1280, 720), color=(255, 255, 255))  # White background
            draw = ImageDraw.Draw(image)
            font = ImageFont.load_default()  # You can specify a different font here
            draw.text((50, 200), line, fill="black", font=font)  # Positioning text on image
            
            # Convert the PIL image to an OpenCV format (BGR)
            opencv_image = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
            images.append(opencv_image)
            
            # Timings for each caption, assuming each caption stays for a fixed duration
            timings.append(time.time() + idx * image_duration)
        
        # Create a video from the images and captions
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Define video codec (MP4)
        video_out = cv2.VideoWriter(video_output_path, fourcc, 1.0, (1280, 720))  # 1 FPS to match 1-second per slide
        
        for img in images:
            video_out.write(img)  # Write each image to the video
            
        # Close video writer
        video_out.release()
        print(f"Video created successfully: {video_output_path}")
        
        # Add audio to the video
        add_audio_to_video(video_output_path, audio_output_path)
    
    except Exception as e:
        print(f"Error during video creation: {e}")

# Function to add audio to the video (using pydub and ffmpeg)
def add_audio_to_video(video_path, audio_path):
    try:
        # Convert the audio file to a format supported by ffmpeg (e.g., WAV if necessary)
        audio = AudioSegment.from_mp3(audio_path)
        audio.export("temp_audio.wav", format="wav")

        # Using ffmpeg to add audio to video
        os.system(f"ffmpeg -i {video_path} -i temp_audio.wav -c:v copy -c:a aac -strict experimental {video_path.replace('.mp4', '_with_audio.mp4')}")

        print("Audio added to the video successfully.")
        
        # Clean up temporary audio file
        os.remove("temp_audio.wav")
    except Exception as e:
        print(f"Error during audio addition: {e}")

# Function to generate image (Optional) - This will be used for each frame in the video
def generate_image_for_caption(caption_text):
    try:
        image = Image.new('RGB', (1280, 720), color=(255, 255, 255))  # White background
        draw = ImageDraw.Draw(image)
        font = ImageFont.load_default()  # You can specify a different font here
        draw.text((50, 300), caption_text, fill="black", font=font)  # Positioning text on image
        return image
    except Exception as e:
        print(f"Error generating image for caption: {e}")
        return None

# Main function to execute the process
def main(pdf_path, output_audio_path, output_video_path):
    text = extract_text_from_pdf(pdf_path)
    if text:
        print("Text extracted successfully, now converting to speech...")
        text_to_speech(text, output_audio_path)
        
        print("Creating video from text with captions and images...")
        create_video_from_images_with_captions(text, output_video_path, output_audio_path)
    else:
        print("No text found in the PDF.")

# File paths
pdf_path = r"C:\Users\ASUS\Downloads\Graph Theory (Part 03) (Lec 09) _ Class Notes.pdf"
audio_file_path = r"C:\Users\ASUS\Downloads\output_audio.mp3"
video_file_path = r"C:\Users\ASUS\Downloads\output_video.mp4"

# Check if the PDF file exists before proceeding
if not os.path.exists(pdf_path):
    print(f"Error: The PDF file at {pdf_path} does not exist.")
else:
    # Execute the main function
    main(pdf_path, audio_file_path, video_file_path)


Text extracted successfully, now converting to speech...
Text-to-speech conversion completed and audio file saved.
Creating video from text with captions and images...
Video created successfully: C:\Users\ASUS\Downloads\output_video.mp4
Error during audio addition: [WinError 2] The system cannot find the file specified


In [11]:
import pdfplumber
import pytesseract
from PIL import Image, ImageDraw, ImageFont
import pyttsx3
import os
import cv2
import numpy as np
from pydub import AudioSegment
import time

# Set the path to your Tesseract executable
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"  # Update this path

# Function to extract text from the PDF using pdfplumber or OCR if text is not extractable
def extract_text_from_pdf(pdf_path):
    try:
        text = ''
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages):
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
                else:
                    print(f"Warning: No text extracted from page {page_num + 1}, using OCR...")
                    page_image = page.to_image()
                    page_text = pytesseract.image_to_string(page_image.original)
                    text += page_text + "\n"
        return text
    except FileNotFoundError:
        print(f"Error: The PDF file at {pdf_path} was not found.")
        return ""
    except Exception as e:
        print(f"Error extracting text from PDF: {e}")
        return ""

# Function to convert text to speech and save as an audio file
def text_to_speech(text, output_audio_path):
    try:
        engine = pyttsx3.init()
        engine.setProperty('rate', 150)
        engine.setProperty('volume', 1)
        
        # Save speech to an audio file
        engine.save_to_file(text, output_audio_path)
        engine.runAndWait()
        print("Text-to-speech conversion completed and audio file saved.")
    except Exception as e:
        print(f"Error during text-to-speech conversion: {e}")

# Function to create video from images using OpenCV and add captions
def create_video_from_images_with_captions(text, video_output_path, audio_output_path, image_duration=3):
    try:
        # Split text into segments (for simplicity, each line is a segment)
        text_lines = text.split('\n')
        
        # Create the images for each line of text (captions)
        images = []
        timings = []
        for idx, line in enumerate(text_lines):
            if line.strip() == "":
                continue  # Skip empty lines
            
            # Generate the image with the caption
            image = generate_image_for_caption(line)
            if image:
                # Convert the PIL image to an OpenCV format (BGR)
                opencv_image = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
                images.append(opencv_image)
            
            # Timings for each caption, assuming each caption stays for a fixed duration
            timings.append(time.time() + idx * image_duration)
        
        # Create a video from the images and captions
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Define video codec (MP4)
        video_out = cv2.VideoWriter(video_output_path, fourcc, 1.0, (1280, 720))  # 1 FPS to match 1-second per slide
        
        for img in images:
            video_out.write(img)  # Write each image to the video
            
        # Close video writer
        video_out.release()
        print(f"Video created successfully: {video_output_path}")
        
        # Add audio to the video
        add_audio_to_video(video_output_path, audio_output_path)
    
    except Exception as e:
        print(f"Error during video creation: {e}")

# Function to add audio to the video (using pydub and ffmpeg)
def add_audio_to_video(video_path, audio_path):
    try:
        # Convert the audio file to a format supported by ffmpeg (e.g., WAV if necessary)
        audio = AudioSegment.from_mp3(audio_path)
        audio.export("temp_audio.wav", format="wav")

        # Using ffmpeg to add audio to video
        os.system(f"ffmpeg -i {video_path} -i temp_audio.wav -c:v copy -c:a aac -strict experimental {video_path.replace('.mp4', '_with_audio.mp4')}")

        print("Audio added to the video successfully.")
        
        # Clean up temporary audio file
        os.remove("temp_audio.wav")
    except Exception as e:
        print(f"Error during audio addition: {e}")

# Function to generate image (Optional) - This will be used for each frame in the video
def generate_image_for_caption(caption_text):
    try:
        # Create a blank white image
        image = Image.new('RGB', (1280, 720), color=(255, 255, 255))  # White background
        draw = ImageDraw.Draw(image)
        
        # Specify the font path (you need the font file for Comic Sans)
        font_path = r"C:\Windows\Fonts\comic.ttf"  # Update the path to the Comic Sans font
        font_size = 60  # Increase the font size
        font = ImageFont.truetype(font_path, font_size)  # Load the Comic Sans font with the new size
        
        # Draw the text on the image, centering it
        text_width, text_height = draw.textsize(caption_text, font=font)
        position = ((1280 - text_width) // 2, (720 - text_height) // 2)  # Center the text
        draw.text(position, caption_text, fill="black", font=font)  # Draw the text in black color
        
        return image
    except Exception as e:
        print(f"Error generating image for caption: {e}")
        return None

# Main function to execute the process
def main(pdf_path, output_audio_path, output_video_path):
    text = extract_text_from_pdf(pdf_path)
    if text:
        print("Text extracted successfully, now converting to speech...")
        text_to_speech(text, output_audio_path)
        
        print("Creating video from text with captions and images...")
        create_video_from_images_with_captions(text, output_video_path, output_audio_path)
    else:
        print("No text found in the PDF.")

# File paths
pdf_path = r"C:\Users\ASUS\Downloads\Graph Theory (Part 03) (Lec 09) _ Class Notes.pdf"
audio_file_path = r"C:\Users\ASUS\Downloads\output_audio.mp3"
video_file_path = r"C:\Users\ASUS\Downloads\output_video.mp4"

# Check if the PDF file exists before proceeding
if not os.path.exists(pdf_path):
    print(f"Error: The PDF file at {pdf_path} does not exist.")
else:
    # Execute the main function
    main(pdf_path, audio_file_path, video_file_path)


Text extracted successfully, now converting to speech...
Text-to-speech conversion completed and audio file saved.
Creating video from text with captions and images...
Error generating image for caption: 'ImageDraw' object has no attribute 'textsize'
Error generating image for caption: 'ImageDraw' object has no attribute 'textsize'
Error generating image for caption: 'ImageDraw' object has no attribute 'textsize'
Error generating image for caption: 'ImageDraw' object has no attribute 'textsize'
Error generating image for caption: 'ImageDraw' object has no attribute 'textsize'
Error generating image for caption: 'ImageDraw' object has no attribute 'textsize'
Error generating image for caption: 'ImageDraw' object has no attribute 'textsize'
Error generating image for caption: 'ImageDraw' object has no attribute 'textsize'
Error generating image for caption: 'ImageDraw' object has no attribute 'textsize'
Error generating image for caption: 'ImageDraw' object has no attribute 'textsize'
Er

In [12]:
import pdfplumber
import pytesseract
from PIL import Image, ImageDraw, ImageFont
import pyttsx3
import os
import cv2
import numpy as np
from pydub import AudioSegment
import time

# Set the path to your Tesseract executable
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"  # Update this path

# Function to extract text from the PDF using pdfplumber or OCR if text is not extractable
def extract_text_from_pdf(pdf_path):
    try:
        text = ''
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages):
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
                else:
                    print(f"Warning: No text extracted from page {page_num + 1}, using OCR...")
                    page_image = page.to_image()
                    page_text = pytesseract.image_to_string(page_image.original)
                    text += page_text + "\n"
        return text
    except FileNotFoundError:
        print(f"Error: The PDF file at {pdf_path} was not found.")
        return ""
    except Exception as e:
        print(f"Error extracting text from PDF: {e}")
        return ""

# Function to convert text to speech and save as an audio file
def text_to_speech(text, output_audio_path):
    try:
        engine = pyttsx3.init()
        engine.setProperty('rate', 150)
        engine.setProperty('volume', 1)
        
        # Save speech to an audio file
        engine.save_to_file(text, output_audio_path)
        engine.runAndWait()
        print("Text-to-speech conversion completed and audio file saved.")
    except Exception as e:
        print(f"Error during text-to-speech conversion: {e}")

# Function to create video from images using OpenCV and add captions
def create_video_from_images_with_captions(text, video_output_path, audio_output_path, image_duration=3):
    try:
        # Split text into segments (for simplicity, each line is a segment)
        text_lines = text.split('\n')
        
        # Create the images for each line of text (captions)
        images = []
        timings = []
        for idx, line in enumerate(text_lines):
            if line.strip() == "":
                continue  # Skip empty lines
            
            # Generate the image with the caption
            image = generate_image_for_caption(line)
            if image:
                # Convert the PIL image to an OpenCV format (BGR)
                opencv_image = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
                images.append(opencv_image)
            
            # Timings for each caption, assuming each caption stays for a fixed duration
            timings.append(time.time() + idx * image_duration)
        
        # Create a video from the images and captions
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Define video codec (MP4)
        video_out = cv2.VideoWriter(video_output_path, fourcc, 1.0, (1280, 720))  # 1 FPS to match 1-second per slide
        
        for img in images:
            video_out.write(img)  # Write each image to the video
            
        # Close video writer
        video_out.release()
        print(f"Video created successfully: {video_output_path}")
        
        # Add audio to the video
        add_audio_to_video(video_output_path, audio_output_path)
    
    except Exception as e:
        print(f"Error during video creation: {e}")

# Function to add audio to the video (using pydub and ffmpeg)
def add_audio_to_video(video_path, audio_path):
    try:
        # Convert the audio file to a format supported by ffmpeg (e.g., WAV if necessary)
        audio = AudioSegment.from_mp3(audio_path)
        audio.export("temp_audio.wav", format="wav")

        # Using ffmpeg to add audio to video
        os.system(f"ffmpeg -i {video_path} -i temp_audio.wav -c:v copy -c:a aac -strict experimental {video_path.replace('.mp4', '_with_audio.mp4')}")

        print("Audio added to the video successfully.")
        
        # Clean up temporary audio file
        os.remove("temp_audio.wav")
    except Exception as e:
        print(f"Error during audio addition: {e}")

# Function to generate image (Optional) - This will be used for each frame in the video
def generate_image_for_caption(caption_text):
    try:
        # Create a blank white image
        image = Image.new('RGB', (1280, 720), color=(255, 255, 255))  # White background
        draw = ImageDraw.Draw(image)
        
        # Specify the font path (you need the font file for Comic Sans)
        font_path = r"C:\Windows\Fonts\comic.ttf"  # Update the path to the Comic Sans font
        font_size = 60  # Increase the font size
        font = ImageFont.truetype(font_path, font_size)  # Load the Comic Sans font with the new size
        
        # Get text bounding box (for centering)
        bbox = draw.textbbox((0, 0), caption_text, font=font)  # Get the bounding box of the text
        text_width = bbox[2] - bbox[0]  # Width of the text
        text_height = bbox[3] - bbox[1]  # Height of the text
        
        # Calculate position to center the text
        position = ((1280 - text_width) // 2, (720 - text_height) // 2)  # Center the text
        draw.text(position, caption_text, fill="black", font=font)  # Draw the text in black color
        
        return image
    except Exception as e:
        print(f"Error generating image for caption: {e}")
        return None

# Main function to execute the process
def main(pdf_path, output_audio_path, output_video_path):
    text = extract_text_from_pdf(pdf_path)
    if text:
        print("Text extracted successfully, now converting to speech...")
        text_to_speech(text, output_audio_path)
        
        print("Creating video from text with captions and images...")
        create_video_from_images_with_captions(text, output_video_path, output_audio_path)
    else:
        print("No text found in the PDF.")

# File paths
pdf_path = r"C:\Users\ASUS\Downloads\Graph Theory (Part 03) (Lec 09) _ Class Notes.pdf"
audio_file_path = r"C:\Users\ASUS\Downloads\output_audio.mp3"
video_file_path = r"C:\Users\ASUS\Downloads\output_video.mp4"

# Check if the PDF file exists before proceeding
if not os.path.exists(pdf_path):
    print(f"Error: The PDF file at {pdf_path} does not exist.")
else:
    # Execute the main function
    main(pdf_path, audio_file_path, video_file_path)


Text extracted successfully, now converting to speech...
Text-to-speech conversion completed and audio file saved.
Creating video from text with captions and images...
Video created successfully: C:\Users\ASUS\Downloads\output_video.mp4
Error during audio addition: [WinError 2] The system cannot find the file specified


In [13]:
import pdfplumber
import pytesseract
from PIL import Image, ImageDraw, ImageFont
import pyttsx3
import os
import cv2
import numpy as np
from pydub import AudioSegment
import time
import random  # Importing random module to choose random colors

# Set the path to your Tesseract executable
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"  # Update this path

# Function to extract text from the PDF using pdfplumber or OCR if text is not extractable
def extract_text_from_pdf(pdf_path):
    try:
        text = ''
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages):
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
                else:
                    print(f"Warning: No text extracted from page {page_num + 1}, using OCR...")
                    page_image = page.to_image()
                    page_text = pytesseract.image_to_string(page_image.original)
                    text += page_text + "\n"
        return text
    except FileNotFoundError:
        print(f"Error: The PDF file at {pdf_path} was not found.")
        return ""
    except Exception as e:
        print(f"Error extracting text from PDF: {e}")
        return ""

# Function to convert text to speech and save as an audio file
def text_to_speech(text, output_audio_path):
    try:
        engine = pyttsx3.init()
        engine.setProperty('rate', 150)
        engine.setProperty('volume', 1)
        
        # Save speech to an audio file
        engine.save_to_file(text, output_audio_path)
        engine.runAndWait()
        print("Text-to-speech conversion completed and audio file saved.")
    except Exception as e:
        print(f"Error during text-to-speech conversion: {e}")

# Function to create video from images using OpenCV and add captions
def create_video_from_images_with_captions(text, video_output_path, audio_output_path, image_duration=3):
    try:
        # Split text into segments (for simplicity, each line is a segment)
        text_lines = text.split('\n')
        
        # Create the images for each line of text (captions)
        images = []
        timings = []
        for idx, line in enumerate(text_lines):
            if line.strip() == "":
                continue  # Skip empty lines
            
            # Generate the image with the caption
            image = generate_image_for_caption(line)
            if image:
                # Convert the PIL image to an OpenCV format (BGR)
                opencv_image = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
                images.append(opencv_image)
            
            # Timings for each caption, assuming each caption stays for a fixed duration
            timings.append(time.time() + idx * image_duration)
        
        # Create a video from the images and captions
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Define video codec (MP4)
        video_out = cv2.VideoWriter(video_output_path, fourcc, 1.0, (1280, 720))  # 1 FPS to match 1-second per slide
        
        for img in images:
            video_out.write(img)  # Write each image to the video
            
        # Close video writer
        video_out.release()
        print(f"Video created successfully: {video_output_path}")
        
        # Add audio to the video
        add_audio_to_video(video_output_path, audio_output_path)
    
    except Exception as e:
        print(f"Error during video creation: {e}")

# Function to add audio to the video (using pydub and ffmpeg)
def add_audio_to_video(video_path, audio_path):
    try:
        # Convert the audio file to a format supported by ffmpeg (e.g., WAV if necessary)
        audio = AudioSegment.from_mp3(audio_path)
        audio.export("temp_audio.wav", format="wav")

        # Using ffmpeg to add audio to video
        os.system(f"ffmpeg -i {video_path} -i temp_audio.wav -c:v copy -c:a aac -strict experimental {video_path.replace('.mp4', '_with_audio.mp4')}")

        print("Audio added to the video successfully.")
        
        # Clean up temporary audio file
        os.remove("temp_audio.wav")
    except Exception as e:
        print(f"Error during audio addition: {e}")

# Function to generate image (Optional) - This will be used for each frame in the video
def generate_image_for_caption(caption_text):
    try:
        # Create a blank white image
        image = Image.new('RGB', (1280, 720), color=(255, 255, 255))  # White background
        draw = ImageDraw.Draw(image)
        
        # Specify the font path (you need the font file for Comic Sans)
        font_path = r"C:\Windows\Fonts\comic.ttf"  # Update the path to the Comic Sans font
        font_size = 60  # Increase the font size
        font = ImageFont.truetype(font_path, font_size)  # Load the Comic Sans font with the new size
        
        # Get text bounding box (for centering)
        bbox = draw.textbbox((0, 0), caption_text, font=font)  # Get the bounding box of the text
        text_width = bbox[2] - bbox[0]  # Width of the text
        text_height = bbox[3] - bbox[1]  # Height of the text
        
        # Calculate position to center the text
        position = ((1280 - text_width) // 2, (720 - text_height) // 2)  # Center the text
        
        # Random color for each caption (RGB format)
        text_color = random_color()  # Generate a random color for the caption text
        
        draw.text(position, caption_text, fill=text_color, font=font)  # Draw the text in a random color
        
        return image
    except Exception as e:
        print(f"Error generating image for caption: {e}")
        return None

# Function to generate random RGB color
def random_color():
    return tuple(random.randint(0, 255) for _ in range(3))  # Generate a random RGB color

# Main function to execute the process
def main(pdf_path, output_audio_path, output_video_path):
    text = extract_text_from_pdf(pdf_path)
    if text:
        print("Text extracted successfully, now converting to speech...")
        text_to_speech(text, output_audio_path)
        
        print("Creating video from text with captions and images...")
        create_video_from_images_with_captions(text, output_video_path, output_audio_path)
    else:
        print("No text found in the PDF.")

# File paths
pdf_path = r"C:\Users\ASUS\Downloads\Graph Theory (Part 03) (Lec 09) _ Class Notes.pdf"
audio_file_path = r"C:\Users\ASUS\Downloads\output_audio.mp3"
video_file_path = r"C:\Users\ASUS\Downloads\output_video.mp4"

# Check if the PDF file exists before proceeding
if not os.path.exists(pdf_path):
    print(f"Error: The PDF file at {pdf_path} does not exist.")
else:
    # Execute the main function
    main(pdf_path, audio_file_path, video_file_path)


Text extracted successfully, now converting to speech...
Text-to-speech conversion completed and audio file saved.
Creating video from text with captions and images...
Video created successfully: C:\Users\ASUS\Downloads\output_video.mp4
Error during audio addition: [WinError 2] The system cannot find the file specified


In [1]:
pip install matplotlib seaborn manim


  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ---------------------------------------- 0.0/44.9 kB ? eta -:--:--
     ---------------------------------------- 44.9/44.9 kB 2.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/585.2 kB ? eta -:--:--
   ----------- ---------------------------- 174.1/585.2 kB 3.5 MB/s eta 0:00:01
   ----------------------- ---------------- 337.9/585.2 kB 3.5 MB/s eta 0:00:01
   ---------------------------------------  583.7/585.2 kB 4.1 MB/s eta 0:00:01
   ---------------------------------------- 585.2/585.2 kB 4.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/54.4 kB ? eta -:--:--
   ---------------------------------------- 54.4/54.4 kB ? eta 0:00:00
   ---------------------------------------- 0.0/4.1 MB ? eta -:--:--
   -- ------------------------------------- 0.2/4.1 MB 4.8 MB/s eta 0:00:01
   ---- ----------------------------------- 0.5/4.1 MB 5.3 MB/s eta 0

In [6]:
import pdfplumber
import pytesseract
from PIL import Image, ImageDraw, ImageFont
import pyttsx3
import os
import cv2
import numpy as np
import random
from pydub import AudioSegment
import time  # Importing the time module to fix the 'time is not defined' error
import matplotlib.pyplot as plt
import seaborn as sns

# Set the path to your Tesseract executable
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"  # Update this path

# Function to extract text from the PDF using pdfplumber or OCR if text is not extractable
def extract_text_from_pdf(pdf_path):
    try:
        text = ''
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages):
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
                else:
                    print(f"Warning: No text extracted from page {page_num + 1}, using OCR...")
                    page_image = page.to_image()
                    page_text = pytesseract.image_to_string(page_image.original)
                    text += page_text + "\n"
        return text
    except FileNotFoundError:
        print(f"Error: The PDF file at {pdf_path} was not found.")
        return ""
    except Exception as e:
        print(f"Error extracting text from PDF: {e}")
        return ""

# Function to convert text to speech and save as an audio file
def text_to_speech(text, output_audio_path):
    try:
        engine = pyttsx3.init()
        engine.setProperty('rate', 150)
        engine.setProperty('volume', 1)
        
        # Save speech to an audio file
        engine.save_to_file(text, output_audio_path)
        engine.runAndWait()
        print("Text-to-speech conversion completed and audio file saved.")
    except Exception as e:
        print(f"Error during text-to-speech conversion: {e}")

# Function to generate graphs/charts for visual representation
def generate_graphs():
    # Example of creating a simple bar chart
    categories = ['A', 'B', 'C', 'D', 'E']
    values = np.random.randint(1, 10, size=5)

    plt.figure(figsize=(5, 3))
    sns.barplot(x=categories, y=values, palette='Blues')
    plt.title('Sample Bar Chart')
    plt.ylabel('Values')
    
    # Save the plot as an image
    graph_path = "sample_graph.png"
    plt.savefig(graph_path)
    plt.close()
    return graph_path

# Function to create video from images using OpenCV and add captions
def create_video_from_images_with_captions(text, video_output_path, audio_output_path, image_duration=3):
    try:
        # Split text into segments (for simplicity, each line is a segment)
        text_lines = text.split('\n')
        
        # Create the images for each line of text (captions)
        images = []
        timings = []
        for idx, line in enumerate(text_lines):
            if line.strip() == "":
                continue  # Skip empty lines
            
            # Generate the image with the caption
            image = generate_image_for_caption(line)
            if image:
                # Convert the PIL image to an OpenCV format (BGR)
                opencv_image = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
                images.append(opencv_image)
            
            # Timings for each caption, assuming each caption stays for a fixed duration
            timings.append(time.time() + idx * image_duration)
        
        # Generate and add graphs/charts as additional images
        graph_path = generate_graphs()
        graph_image = cv2.imread(graph_path)
        images.append(graph_image)
        
        # Create a video from the images and captions
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Define video codec (MP4)
        video_out = cv2.VideoWriter(video_output_path, fourcc, 1.0, (1280, 720))  # 1 FPS to match 1-second per slide
        
        for img in images:
            video_out.write(img)  # Write each image to the video
            
        # Close video writer
        video_out.release()
        print(f"Video created successfully: {video_output_path}")
        
        # Add audio to the video
        add_audio_to_video(video_output_path, audio_output_path)
    
    except Exception as e:
        print(f"Error during video creation: {e}")

# Function to add audio to the video (using pydub and ffmpeg)
def add_audio_to_video(video_path, audio_path):
    try:
        # Convert the audio file to a format supported by ffmpeg (e.g., WAV if necessary)
        audio = AudioSegment.from_mp3(audio_path)
        audio.export("temp_audio.wav", format="wav")

        # Using ffmpeg to add audio to video
        os.system(f"ffmpeg -i {video_path} -i temp_audio.wav -c:v copy -c:a aac -strict experimental {video_path.replace('.mp4', '_with_audio.mp4')}")

        print("Audio added to the video successfully.")
        
        # Clean up temporary audio file
        os.remove("temp_audio.wav")
    except Exception as e:
        print(f"Error during audio addition: {e}")

# Function to generate image (Optional) - This will be used for each frame in the video
def generate_image_for_caption(caption_text):
    try:
        # Create a blank white image
        image = Image.new('RGB', (1280, 720), color=(255, 255, 255))  # White background
        draw = ImageDraw.Draw(image)
        
        # Specify the font path (you need the font file for Comic Sans)
        font_path = r"C:\Windows\Fonts\comic.ttf"  # Update the path to the Comic Sans font
        font_size = 60  # Increase the font size
        font = ImageFont.truetype(font_path, font_size)  # Load the Comic Sans font with the new size
        
        # Get text bounding box (for centering)
        bbox = draw.textbbox((0, 0), caption_text, font=font)  # Get the bounding box of the text
        text_width = bbox[2] - bbox[0]  # Width of the text
        text_height = bbox[3] - bbox[1]  # Height of the text
        
        # Calculate position to center the text
        position = ((1280 - text_width) // 2, (720 - text_height) // 2)  # Center the text
        
        # Random color for each caption (RGB format)
        text_color = random_color()  # Generate a random color for the caption text
        
        draw.text(position, caption_text, fill=text_color, font=font)  # Draw the text in a random color
        
        return image
    except Exception as e:
        print(f"Error generating image for caption: {e}")
        return None

# Function to generate random RGB color
def random_color():
    return tuple(random.randint(0, 255) for _ in range(3))  # Generate a random RGB color

# Main function to execute the process
def main(pdf_path, output_audio_path, output_video_path):
    text = extract_text_from_pdf(pdf_path)
    if text:
        print("Text extracted successfully, now converting to speech...")
        text_to_speech(text, output_audio_path)
        
        print("Creating video from text with captions and images...")
        create_video_from_images_with_captions(text, output_video_path, output_audio_path)
    else:
        print("No text found in the PDF.")

# File paths
pdf_path = r"C:\Users\ASUS\Downloads\Graph Theory (Part 03) (Lec 09) _ Class Notes.pdf"
audio_file_path = r"C:\Users\ASUS\Downloads\output_audio.mp3"
video_file_path = r"C:\Users\ASUS\Downloads\output_video.mp4"

# Check if the PDF file exists before proceeding
if not os.path.exists(pdf_path):
    print(f"Error: The PDF file at {pdf_path} does not exist.")
else:
    # Execute the main function
    main(pdf_path, audio_file_path, video_file_path)


Text extracted successfully, now converting to speech...
Text-to-speech conversion completed and audio file saved.
Creating video from text with captions and images...


C:\Users\ASUS\anaconda3\Lib\site-packages\seaborn\_oldcore.py:1765: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  order = pd.unique(vector)


Video created successfully: C:\Users\ASUS\Downloads\output_video.mp4
Error during audio addition: [WinError 2] The system cannot find the file specified


C:\Users\ASUS\anaconda3\Lib\site-packages\pydub\utils.py:198: RuntimeWarning: Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work
  warn("Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work", RuntimeWarning)


In [9]:
import pdfplumber
import pytesseract
from PIL import Image, ImageDraw, ImageFont
import pyttsx3
import os
import cv2
import numpy as np
import random
from pydub import AudioSegment
import time  # Importing the time module to fix the 'time is not defined' error
import matplotlib.pyplot as plt
import seaborn as sns

# Set the path to your Tesseract executable
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"  # Update this path

# Function to extract text from the PDF using pdfplumber or OCR if text is not extractable
def extract_text_from_pdf(pdf_path):
    try:
        text = ''
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages):
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
                else:
                    print(f"Warning: No text extracted from page {page_num + 1}, using OCR...")
                    page_image = page.to_image()
                    page_text = pytesseract.image_to_string(page_image.original)
                    text += page_text + "\n"
        return text
    except FileNotFoundError:
        print(f"Error: The PDF file at {pdf_path} was not found.")
        return ""
    except Exception as e:
        print(f"Error extracting text from PDF: {e}")
        return ""

# Function to convert text to speech and save as an audio file
def text_to_speech(text, output_audio_path):
    try:
        engine = pyttsx3.init()
        engine.setProperty('rate', 150)
        engine.setProperty('volume', 1)
        
        # Save speech to an audio file
        engine.save_to_file(text, output_audio_path)
        engine.runAndWait()
        print("Text-to-speech conversion completed and audio file saved.")
    except Exception as e:
        print(f"Error during text-to-speech conversion: {e}")

# Function to generate graphs/charts for visual representation
def generate_graphs():
    # Example of creating a simple bar chart
    categories = ['A', 'B', 'C', 'D', 'E']
    values = np.random.randint(1, 10, size=5)

    plt.figure(figsize=(5, 3))
    sns.barplot(x=categories, y=values, palette='Blues')
    plt.title('Sample Bar Chart')
    plt.ylabel('Values')
    
    # Save the plot as an image
    graph_path = "sample_graph.png"
    plt.savefig(graph_path)
    plt.close()
    return graph_path

# Function to create video from images using OpenCV and add captions
def create_video_from_images_with_captions(text, video_output_path, audio_output_path, image_duration=3):
    try:
        # Split text into segments (for simplicity, each line is a segment)
        text_lines = text.split('\n')
        
        # Create the images for each line of text (captions)
        images = []
        timings = []
        for idx, line in enumerate(text_lines):
            if line.strip() == "":
                continue  # Skip empty lines
            
            # Generate the image with the caption
            image = generate_image_for_caption(line)
            if image:
                # Convert the PIL image to an OpenCV format (BGR)
                opencv_image = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
                images.append(opencv_image)
            
            # Timings for each caption, assuming each caption stays for a fixed duration
            timings.append(time.time() + idx * image_duration)
        
        # Generate and add graphs/charts as additional images
        graph_path = generate_graphs()
        graph_image = cv2.imread(graph_path)
        images.append(graph_image)
        
        # Create a video from the images and captions
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Define video codec (MP4)
        video_out = cv2.VideoWriter(video_output_path, fourcc, 1.0, (1280, 720))  # 1 FPS to match 1-second per slide
        
        for img in images:
            video_out.write(img)  # Write each image to the video
            
        # Close video writer
        video_out.release()
        print(f"Video created successfully: {video_output_path}")
        
        # Add audio to the video
        add_audio_to_video(video_output_path, audio_output_path)
    
    except Exception as e:
        print(f"Error during video creation: {e}")

# Function to add audio to the video (using pydub and ffmpeg)
def add_audio_to_video(video_path, audio_path):
    try:
        # Convert the audio file to a format supported by ffmpeg (e.g., WAV if necessary)
        audio = AudioSegment.from_mp3(audio_path)
        audio.export("temp_audio.wav", format="wav")

        # Using ffmpeg to add audio to video
        os.system(f"ffmpeg -i {video_path} -i temp_audio.wav -c:v copy -c:a aac -strict experimental {video_path.replace('.mp4', '_with_audio.mp4')}")

        print("Audio added to the video successfully.")
        
        # Clean up temporary audio file
        os.remove("temp_audio.wav")
    except Exception as e:
        print(f"Error during audio addition: {e}")

# Function to generate image (Optional) - This will be used for each frame in the video
def generate_image_for_caption(caption_text):
    try:
        # Generate random text color
        text_color = random_color()  # Generate a random RGB color for the text
        
        # Compute a complementary background color for the contrast
        background_color = get_contrasting_background(text_color)
        
        # Create a blank image with the computed background color
        image = Image.new('RGB', (1280, 720), color=background_color)  # Dynamic background color
        draw = ImageDraw.Draw(image)
        
        # Specify the font path (you need the font file for Comic Sans)
        font_path = r"C:\Windows\Fonts\comic.ttf"  # Update the path to the Comic Sans font
        font_size = 60  # Increase the font size
        font = ImageFont.truetype(font_path, font_size)  # Load the Comic Sans font with the new size
        
        # Get text bounding box (for centering)
        bbox = draw.textbbox((0, 0), caption_text, font=font)  # Get the bounding box of the text
        text_width = bbox[2] - bbox[0]  # Width of the text
        text_height = bbox[3] - bbox[1]  # Height of the text
        
        # Calculate position to center the text
        position = ((1280 - text_width) // 2, (720 - text_height) // 2)  # Center the text
        
        draw.text(position, caption_text, fill=text_color, font=font)  # Draw the text in the random color
        
        return image
    except Exception as e:
        print(f"Error generating image for caption: {e}")
        return None

# Function to generate random RGB color
def random_color():
    return tuple(random.randint(0, 255) for _ in range(3))  # Generate a random RGB color

# Function to compute a contrasting background color
def get_contrasting_background(text_color):
    # Calculate the luminance of the text color
    luminance = 0.2126 * text_color[0] + 0.7152 * text_color[1] + 0.0722 * text_color[2]
    
    # If luminance is high (lighter color), choose dark background; if low, choose light background
    if luminance > 128:
        return (0, 0, 0)  # Dark background for light text
    else:
        return (255, 255, 255)  # Light background for dark text

# Main function to execute the process
def main(pdf_path, output_audio_path, output_video_path):
    text = extract_text_from_pdf(pdf_path)
    if text:
        print("Text extracted successfully, now converting to speech...")
        text_to_speech(text, output_audio_path)
        
        print("Creating video from text with captions and images...")
        create_video_from_images_with_captions(text, output_video_path, output_audio_path)
    else:
        print("No text found in the PDF.")

# File paths
pdf_path = r"C:\Users\ASUS\Downloads\Graph Theory (Part 03) (Lec 09) _ Class Notes.pdf"
audio_file_path = r"C:\Users\ASUS\Downloads\output_audio.mp3"
video_file_path = r"C:\Users\ASUS\Downloads\output_video.mp4"

# Check if the PDF file exists before proceeding
if not os.path.exists(pdf_path):
    print(f"Error: The PDF file at {pdf_path} does not exist.")
else:
    # Execute the main function
    main(pdf_path, audio_file_path, video_file_path)


Text extracted successfully, now converting to speech...
Text-to-speech conversion completed and audio file saved.
Creating video from text with captions and images...


C:\Users\ASUS\anaconda3\Lib\site-packages\seaborn\_oldcore.py:1765: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  order = pd.unique(vector)


Video created successfully: C:\Users\ASUS\Downloads\output_video.mp4
Error during audio addition: [WinError 2] The system cannot find the file specified


C:\Users\ASUS\anaconda3\Lib\site-packages\pydub\utils.py:198: RuntimeWarning: Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work
  warn("Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work", RuntimeWarning)
